In [ ]:
from torch import nn
import torch
from torch.utils.data import Subset, Dataset, DataLoader, random_split
from torchvision import models, tv_tensors
from torchvision.transforms import v2
from PIL import Image
import os
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from torchvision.utils import make_grid
from torch.amp import autocast, GradScaler

In [ ]:
class Resnet(torch.nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.backbone_blocks = nn.ModuleList([self.stem, self.layer1, self.layer2, self.layer3, self.layer4])
        for block in self.backbone_blocks:
            for p in block.parameters():
                p.requires_grad = False

    def unfreeze_block(self, block_idx):
        idx_from_end = len(self.backbone_blocks) - 1 - block_idx
        for p in self.backbone_blocks[idx_from_end].parameters():
            p.requires_grad = True

    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x0, x1, x2, x3, x4]
        

UNFREEZE_SCHEDULE = {
    5: 'layer4',
    10: 'layer3',
    15: 'layer2',
    20: 'layer1',
    25: 'stem',
}

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x


class UNetDecoder(nn.Module):
    def __init__(self, num_classes=19):
        super().__init__()
        self.block4 = DecoderBlock(512, 256, 256) 
        self.block3 = DecoderBlock(256, 128, 128) 
        self.block2 = DecoderBlock(128, 64, 64)   
        self.block1 = DecoderBlock(64, 64, 32)     
        self.final_up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(32, num_classes, 1)

    def forward(self, feats):
    
        x = self.block4(feats['layer4'], feats['layer3'])
        x = self.block3(x, feats['layer2'])
        x = self.block2(x, feats['layer1'])
        x = self.block1(x, feats['stem'])
        x = self.final_up(x)
        return self.classifier(x)

In [ ]:
def build_transforms(image_size=512):
    train_transform = v2.Compose([
        v2.RandomResizedCrop(
            size=image_size,
            scale=(0.5, 2.0),
            ratio=(1.8, 2.2),
            interpolation=v2.InterpolationMode.BILINEAR,
            antialias=True,
        ),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64}, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = v2.Compose([
        v2.Resize(image_size, interpolation=v2.InterpolationMode.BILINEAR, antialias=True),
        v2.CenterCrop(image_size),
        v2.ToDtype({tv_tensors.Image: torch.float32, tv_tensors.Mask: torch.int64}, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return train_transform, val_transform



In [ ]:
CITYSCAPES_LABELID_TO_TRAINID = {
    0: 255, 1: 255, 2: 255, 3: 255, 4: 255, 5: 255, 6: 255,
    7: 0,   # road
    8: 1,   # sidewalk
    9: 255, 10: 255,
    11: 2,  # building
    12: 3,  # wall
    13: 4,  # fence
    14: 255, 15: 255, 16: 255,
    17: 5,  # pole
    18: 255,
    19: 6,  # traffic light
    20: 7,  # traffic sign
    21: 8,  # vegetation
    22: 9,  # terrain
    23: 10, # sky
    24: 11, # person
    25: 12, # rider
    26: 13, # car
    27: 14, # truck
    28: 15, # bus
    29: 255, 30: 255,
    31: 16, # train
    32: 17, # motorcycle
    33: 18, # bicycle
    -1: 255,
}

CLASS_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic_light', 'traffic_sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle'
]
NUM_CLASSES = 19
IGNORE_INDEX = 255

_LUT = np.full(256, IGNORE_INDEX, dtype=np.uint8)
for k, v in CITYSCAPES_LABELID_TO_TRAINID.items():
    if k >= 0:
        _LUT[k] = v

class CityscapesDataset(Dataset):
    
    def __init__(self, root, split='train', transform=None):
        self.root = root
        self.split = split
        self.transform = transform

        img_dir = os.path.join(root, 'leftImg8bit', split)
        mask_dir = os.path.join(root, 'gtFine', split)

        self.images, self.masks = [], []
        for city in sorted(os.listdir(img_dir)):
            city_img_dir = os.path.join(img_dir, city)
            city_mask_dir = os.path.join(mask_dir, city)
            for fname in sorted(os.listdir(city_img_dir)):
                if not fname.endswith('_leftImg8bit.png'):
                    continue
                mask_fname = fname.replace('_leftImg8bit.png', '_gtFine_labelIds.png')
                self.images.append(os.path.join(city_img_dir, fname))
                self.masks.append(os.path.join(city_mask_dir, mask_fname))

        assert len(self.images) == len(self.masks) and len(self.images) > 0, \
            f"Не найдены пары image/mask в {root} для split={split}"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        mask = Image.open(self.masks[idx])

        img = tv_tensors.Image(img)
        mask = tv_tensors.Mask(np.array(mask, dtype=np.int64))

        if self.transform is not None:
            img, mask = self.transform(img, mask)

    
        mask_np = mask.numpy().astype(np.uint8)
        mask_np = _LUT[mask_np]
        mask = torch.as_tensor(mask_np, dtype=torch.long)

        return img, mask


def build_dataloaders(root, image_size=512, batch_size=8, num_workers=0):
    train_tf, val_tf = build_transforms(image_size=image_size)

    train_ds = CityscapesDataset(root, split='train', transform=train_tf)
    val_ds = CityscapesDataset(root, split='val', transform=val_tf)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, val_loader

In [ ]:
class IoUMeter:
    def __init__(self, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.confmat = torch.zeros(num_classes, num_classes, dtype=torch.int64)

    def update(self, pred, target):
        mask = target != self.ignore_index
        pred = pred[mask]
        target = target[mask]
        idx = target * self.num_classes + pred
        idx = idx.clamp(0, self.num_classes ** 2 - 1)
        binc = torch.bincount(idx, minlength=self.num_classes ** 2)
        self.confmat += binc.reshape(self.num_classes, self.num_classes).cpu()

    def compute(self):
        cm = self.confmat.float()
        intersection = torch.diag(cm)
        union = cm.sum(0) + cm.sum(1) - intersection
        iou = intersection / union.clamp(min=1)
        miou = iou[union > 0].mean().item()
        return miou, iou

    def reset(self):
        self.confmat.zero_()


def build_optimizer(model, base_lr=1e-3, encoder_lr_mult=0.1):
    encoder_params = list(model.encoder.parameters())
    decoder_params = list(model.decoder.parameters())

    optimizer = torch.optim.AdamW([
        {'params': [p for p in encoder_params if p.requires_grad], 'lr': base_lr * encoder_lr_mult},
        {'params': decoder_params, 'lr': base_lr},
    ], weight_decay=1e-4)
    return optimizer

In [ ]:
# цвета для раскраски маски
CITYSCAPES_PALETTE = torch.tensor([
    [128, 64, 128],   # road
    [244, 35, 232],   # sidewalk
    [70, 70, 70],     # building
    [102, 102, 156],  # wall
    [190, 153, 153],  # fence
    [153, 153, 153],  # pole
    [250, 170, 30],   # traffic light
    [220, 220, 0],    # traffic sign
    [107, 142, 35],   # vegetation
    [152, 251, 152],  # terrain
    [70, 130, 180],   # sky
    [220, 20, 60],    # person
    [255, 0, 0],      # rider
    [0, 0, 142],      # car
    [0, 0, 70],       # truck
    [0, 60, 100],      # bus
    [0, 80, 100],     # train
    [0, 0, 230],      # motorcycle
    [119, 11, 32],    # bicycle
], dtype=torch.uint8)

def colorize_mask(mask, palette=CITYSCAPES_PALETTE, ignore_index=IGNORE_INDEX):
    """mask: [H, W] long tensor с trainId (0..18) или ignore_index -> [3, H, W] uint8"""
    h, w = mask.shape
    color = torch.zeros(3, h, w, dtype=torch.uint8)
    for cls_id in range(len(palette)):
        m = mask == cls_id
        for c in range(3):
            color[c][m] = palette[cls_id][c]
    # ignore_index оставляем чёрным (уже 0,0,0 по умолчанию)
    return color

def denormalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """img: [3, H, W] float tensor, нормализованный -> [3, H, W] uint8 в диапазоне [0,255]"""
    mean = torch.tensor(mean).view(3, 1, 1).to(img.device)
    std = torch.tensor(std).view(3, 1, 1).to(img.device)
    img = img * std + mean
    img = (img.clamp(0, 1) * 255).to(torch.uint8)
    return img


def log_prediction_grid(writer, model, fixed_imgs, fixed_masks, epoch, device, use_amp=True):
    """fixed_imgs, fixed_masks: батч из 5 фиксированных val-примеров (уже на CPU)"""
    model.eval()
    with torch.no_grad():
        imgs = fixed_imgs.to(device)
        with autocast(device_type='cuda', enabled=use_amp):
            logits = model(imgs)
            logits = nn.functional.interpolate(
                logits, size=fixed_masks.shape[-2:], mode='bilinear', align_corners=False
            )
        preds = logits.float().argmax(1).cpu()

    rows = []
    for i in range(imgs.shape[0]):
        orig = denormalize(fixed_imgs[i])
        gt_color = colorize_mask(fixed_masks[i])
        pred_color = colorize_mask(preds[i])
        # кладём по горизонтали: original | GT | prediction
        row = torch.cat([orig, gt_color, pred_color], dim=2)  # concat по width
        rows.append(row)

    grid = make_grid(rows, nrow=1, padding=4)  # каждая строка = один пример, по вертикали 5 строк
    writer.add_image('val/predictions_grid', grid, epoch)
    model.train()

In [ ]:
from torch.utils.tensorboard import SummaryWriter

def train_model(model, train_loader, val_loader, num_epochs=30, device='cuda',
                 base_lr=1e-3, unfreeze_schedule=UNFREEZE_SCHEDULE, use_amp=True,
                 log_dir='RUNS_SEGM'):
    model.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.decoder.parameters(), lr=base_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = GradScaler(device='cuda', enabled=use_amp)
    writer = SummaryWriter(log_dir=log_dir)

    # --- фиксируем 5 примеров из val для визуализации на протяжении всего обучения ---
    fixed_imgs, fixed_masks = next(iter(val_loader))
    fixed_imgs, fixed_masks = fixed_imgs[:5], fixed_masks[:5]

    best_miou = 0.0
    global_step = 0
    log_every_n_epochs = 4
    epoch_no_improvment = 0

    for epoch in range(1, num_epochs + 1):
        if epoch in unfreeze_schedule:
            stage_name = unfreeze_schedule[epoch]
            model.encoder.unfreeze_block(stage_name)
            optimizer.add_param_group({
                'params': [p for p in getattr(model.encoder, stage_name).parameters()],
                'lr': base_lr * 0.1,
            })
            print(f"[epoch {epoch}] Разморожен {stage_name}, добавлена param_group")
            writer.add_text('training/unfreeze_events', f"Epoch {epoch}: unfroze {stage_name}", epoch)

        model.train()
        for name in ['stem', 'layer1', 'layer2', 'layer3', 'layer4']:
            stage = getattr(model.encoder, name)
            if not next(stage.parameters()).requires_grad:
                stage.eval()

        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} [train]")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast(device_type='cuda', enabled=use_amp):
                logits = model(imgs)
                logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            pbar.set_postfix(loss=running_loss / (pbar.n + 1))

            # --- логируем loss каждый шаг (полезно видеть шум внутри эпохи) ---
            writer.add_scalar('train/loss_step', loss.item(), global_step)
            global_step += 1

        scheduler.step()
        avg_train_loss = running_loss / len(train_loader)

        # --- val ---
        model.eval()
        iou_meter = IoUMeter()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{num_epochs} [val]"):
                imgs, masks = imgs.to(device), masks.to(device)
                with autocast(device_type='cuda', enabled=use_amp):
                    logits = model(imgs)
                    logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
                    loss = criterion(logits, masks)
                val_loss += loss.item()
                preds = logits.float().argmax(1)
                iou_meter.update(preds, masks)

        avg_val_loss = val_loss / len(val_loader)
        miou, per_class_iou = iou_meter.compute()

        # --- скаляры за эпоху ---
        writer.add_scalar('train/loss_epoch', avg_train_loss, epoch)
        writer.add_scalar('val/loss_epoch', avg_val_loss, epoch)
        writer.add_scalar('val/mIoU', miou, epoch)
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            writer.add_scalar(f'val/IoU_per_class/{cls_name}', per_class_iou[cls_idx].item(), epoch)
        for i, group in enumerate(optimizer.param_groups):
            writer.add_scalar(f'lr/group_{i}', group['lr'], epoch)

        if epoch % log_every_n_epochs == 0: 
            log_prediction_grid(writer, model, fixed_imgs, fixed_masks, epoch, device, use_amp)

        print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} mIoU={miou:.4f}")

        if miou > best_miou:
            best_miou = miou
            epoch_no_improvment = 0
            torch.save(model.state_dict(), 'best_model.pt')
            print(f"  -> сохранена лучшая модель (mIoU={miou:.4f})")

        else:
            torch.save(model.state_dict(), 'last_model.pt')
            epoch_no_improvment += 1
            
    writer.close()
    return model

In [ ]:
class ResNetUNet(nn.Module):
    """Полная модель: encoder + decoder, с единой точкой forward."""
    def __init__(self, num_classes=19, pretrained=True):
        super().__init__()
        self.encoder = Resnet(pretrained=pretrained)
        self.decoder = UNetDecoder(num_classes=num_classes)

    def forward(self, x):
        feats = self.encoder(x)
        out = self.decoder(feats)
        return out

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_loader, val_loader = build_dataloaders(
    root='DATASET',
    image_size=1024,
    batch_size=16,
    num_workers=4,
)

model = ResNetUNet(num_classes=19, pretrained=True)

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=200,
    device=device,
    base_lr=1.5e-5,
    use_amp=True,
    log_dir='SEGMENT',
    )


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shuvoalok/cityscapes")

print("Path to dataset files:", path)

/home/morozzz/Desktop/detection/CNN/myvenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 199M/199M [00:56<00:00, 3.71MB/s] 

Extracting files...


Path to dataset files: /home/morozzz/.cache/kagglehub/datasets/shuvoalok/cityscapes/versions/2
